# DBA MCP Demo — 3 Models → 1 MCP → 3 Tools (Colab + Gradio)

This notebook demonstrates an **enterprise-style Model Context Protocol (MCP) control plane**:

- **Three LLMs** (or three Azure deployments) on the left
- **One MCP-like control plane** in the middle (a central tool registry)
- **Three tools** on the right:
  - `db_query` (HR DB)
  - `docs_search` (PRD/docs)
  - `weather_lookup` (ops/weather stub)

Any model can call any tool, but **only through MCP** — matching your slide:
> models → MCP → tools (m + n connections)


Install Dependencies

In [1]:
!pip -q install openai gradio pandas


Load Secrets (Colab Secrets → env → fallback)

In [2]:
import os, json

# --- Try Google Colab Secrets first (Runtime → Secrets) ---
def colab_user_secret(name):
    try:
        from google.colab import userdata  # only exists in Colab
        return userdata.get(name)
    except Exception:
        return None

def get_secret(name, default=None):
    return (
        colab_user_secret(name)
        or os.environ.get(name)
        or default
    )

# Set this in Colab Secrets: USE_AZURE = "true" or "false"
USE_AZURE = str(get_secret("USE_AZURE", "false")).lower() in ("1", "true", "yes")

# OpenAI (non-Azure)
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")

# Azure OpenAI (if you prefer real enterprise deployment style)
AZURE_OPENAI_ENDPOINT = get_secret("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = get_secret("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = get_secret("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

# We'll use three deployments for Azure case (set in Colab Secrets)
AZURE_DEPLOYMENT_MODEL1 = get_secret("AZURE_DEPLOYMENT_MODEL1")  # e.g. "gpt-4o-hr"
AZURE_DEPLOYMENT_MODEL2 = get_secret("AZURE_DEPLOYMENT_MODEL2")  # e.g. "gpt-4o-prd"
AZURE_DEPLOYMENT_MODEL3 = get_secret("AZURE_DEPLOYMENT_MODEL3")  # e.g. "gpt-4o-ops"

print("USE_AZURE:", USE_AZURE)
print("OPENAI_API_KEY present:", bool(OPENAI_API_KEY))
print("AZURE_OPENAI_ENDPOINT present:", bool(AZURE_OPENAI_ENDPOINT))
print("AZURE_OPENAI_API_KEY present:", bool(AZURE_OPENAI_API_KEY))

if USE_AZURE:
    print("Azure deployments:",
          AZURE_DEPLOYMENT_MODEL1,
          AZURE_DEPLOYMENT_MODEL2,
          AZURE_DEPLOYMENT_MODEL3)


USE_AZURE: False
OPENAI_API_KEY present: True
AZURE_OPENAI_ENDPOINT present: False
AZURE_OPENAI_API_KEY present: True


Connect LLM Client & Register 3 Actual Models

In [3]:
from openai import OpenAI, AzureOpenAI

if USE_AZURE:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )
    MODEL_REGISTRY = {
        "Model 1 – HR (Azure dep1)": AZURE_DEPLOYMENT_MODEL1,
        "Model 2 – PRD (Azure dep2)": AZURE_DEPLOYMENT_MODEL2,
        "Model 3 – Ops (Azure dep3)": AZURE_DEPLOYMENT_MODEL3,
    }
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    MODEL_REGISTRY = {
        "Model 1 – gpt-4o": "gpt-4o",
        "Model 2 – gpt-4o-mini": "gpt-4o-mini",
        "Model 3 – o3-mini": "o3-mini",  # change if needed
    }

print("MODEL_REGISTRY:")
for label, m in MODEL_REGISTRY.items():
    print(f"  {label} -> {m}")


MODEL_REGISTRY:
  Model 1 – gpt-4o -> gpt-4o
  Model 2 – gpt-4o-mini -> gpt-4o-mini
  Model 3 – o3-mini -> o3-mini


Create Sample Data for Tools

We build a tiny HR DB, docs folder, and weather stub.

In [4]:
import sqlite3, pandas as pd
from pathlib import Path

# 1) SQLite HR DB
db_path = "employees.db"
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("DROP TABLE IF EXISTS employees;")
cur.execute("""
CREATE TABLE employees (
    emp_id   INTEGER PRIMARY KEY,
    name     TEXT,
    dept     TEXT,
    location TEXT,
    email    TEXT
);
""")
rows = [
    (101, "Ravi",  "AI",        "Bengaluru", "ravi@example.com"),
    (102, "Anita", "Analytics", "Hyderabad", "anita@example.com"),
    (103, "Sara",  "Ops",       "Pune",      "sara@example.com"),
]
cur.executemany("INSERT INTO employees VALUES (?,?,?,?,?);", rows)
con.commit()
con.close()

# 2) Docs folder for PRD search
Path("docs").mkdir(exist_ok=True)
Path("docs/PRD.md").write_text(
    "# Payroll Compliance PRD (Excerpt)\n"
    "- Auto-ingest government circulars and update rules.\n"
    "- Compliance Rule Engine with thresholds, slabs, effective dates.\n"
    "- Integration API with payroll engine.\n"
    "- Dashboard: coverage, version history, and anomalies.\n"
)
Path("docs/README.md").write_text(
    "# Internal KB\n"
    "This repo holds excerpts of internal documents to demo the MCP docs tool.\n"
    "- PRD.md — sample product requirement excerpt\n"
    "- Future: Confluence/Jira bridge\n"
)

# 3) Weather stub
weather = pd.DataFrame({
    "city": ["Bengaluru", "Bengaluru", "Hyderabad", "Pune"],
    "date": ["2025-11-16", "2025-11-17", "2025-11-16", "2025-11-16"],
    "forecast": [
        "Sunny with mild breeze",
        "Light showers in evening",
        "Humid with clouds",
        "Clear sky",
    ],
})
weather.to_csv("weather.csv", index=False)

print("Sample data prepared:", db_path, "docs/", "weather.csv")


Sample data prepared: employees.db docs/ weather.csv


MCP Block: Central Tool Registry (3 Tools)

This is your MCP middle block: a single dictionary that knows all tools and their schemas.

In [9]:
from typing import Dict, Any, List
import pandas as pd
import sqlite3
from pathlib import Path

# === MCP: central registry of tools (conceptual MCP server) ===

def tool_employee_lookup(payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Fetch a single employee by emp_id from the employees table.
    Table schema: employees(emp_id, name, dept, location, email)
    """
    emp_id = payload.get("emp_id")
    if emp_id is None:
        return {"error": "emp_id is required"}

    con = sqlite3.connect("employees.db")
    try:
        df = pd.read_sql_query(
            "SELECT emp_id, name, dept, location, email FROM employees WHERE emp_id = ?",
            con,
            params=(emp_id,),
        )
        if df.empty:
            return {"emp_id": emp_id, "error": "Employee not found"}
        return df.to_dict(orient="records")[0]
    except Exception as e:
        return {"emp_id": emp_id, "error": str(e)}
    finally:
        con.close()


def tool_docs_search(payload: Dict[str, Any]) -> Dict[str, Any]:
    query = payload.get("query", "")
    k = int(payload.get("k", 3))
    base = Path("docs")
    hits = []
    for p in base.glob("*.md"):
        text = p.read_text(errors="ignore")
        if query.lower() in text.lower():
            snippet = text[:300]
            hits.append({"file": str(p), "snippet": snippet})
    return {"query": query, "hits": hits[:k]}


def tool_weather_lookup(payload: Dict[str, Any]) -> Dict[str, Any]:
    city = payload.get("city", "")
    date = payload.get("date", "")
    df = pd.read_csv("weather.csv")
    mask = (df["city"].str.lower() == city.lower()) & (df["date"] == date)
    rows = df[mask].to_dict(orient="records")
    if not rows:
        return {"city": city, "date": date, "forecast": "No data in demo store."}
    return rows[0]


MCP_REGISTRY: Dict[str, Dict[str, Any]] = {
    "employee_lookup": {
        "description": (
            "Lookup HR employee details by emp_id in the employees table. "
            "Table columns: emp_id, name, dept, location, email."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "emp_id": {
                    "type": "integer",
                    "description": "Employee ID (emp_id column in the employees table).",
                }
            },
            "required": ["emp_id"],
        },
        "handler": tool_employee_lookup,
    },
    "docs_search": {
        "description": "Search internal docs with a simple keyword contains.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "k": {"type": "integer"},
            },
            "required": ["query"],
        },
        "handler": tool_docs_search,
    },
    "weather_lookup": {
        "description": "Lookup offline weather stub for a given city and date.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "date": {"type": "string"},
            },
            "required": ["city", "date"],
        },
        "handler": tool_weather_lookup,
    },
}


def mcp_list_tools() -> List[Dict[str, Any]]:
    """Return tool definitions with JSON schema for OpenAI function-calling."""
    tools = []
    for name, meta in MCP_REGISTRY.items():
        tools.append({
            "type": "function",
            "function": {
                "name": name,
                "description": meta["description"],
                "parameters": meta["parameters"],
            },
        })
    return tools


def mcp_call_tool(name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    meta = MCP_REGISTRY.get(name)
    if not meta:
        return {"error": f"Unknown tool: {name}"}
    try:
        return meta["handler"](payload)
    except Exception as e:
        return {"error": str(e)}

TOOLS_FOR_OPENAI = mcp_list_tools()
TOOL_NAMES = [t["function"]["name"] for t in TOOLS_FOR_OPENAI]

print("MCP tools registered:", TOOL_NAMES)


MCP tools registered: ['employee_lookup', 'docs_search', 'weather_lookup']


Core Orchestrator: Model + Tool → MCP → Tools

This is the heart: 3 models → one MCP → 3 tools, with tool forcing (or auto).

In [10]:
import json

def ask_with_model_and_tool(
    model_label: str,
    tool_choice_label: str,  # "auto" or one of TOOL_NAMES
    user_msg: str,
    verbose: bool = True,
):
    """
    - model_label: pick LLM from MODEL_REGISTRY
    - tool_choice_label: "auto" (LLM decides) or a specific tool name
    - user_msg: prompt
    Returns: (final_answer, debug_log_str)
    """
    model_name = MODEL_REGISTRY[model_label]
    log_lines = []
    log_lines.append(f"[INFO] Using model: {model_label} -> {model_name}")
    log_lines.append(f"[INFO] MCP tools available: {TOOL_NAMES}")
    log_lines.append(f"[INFO] Tool choice: {tool_choice_label}")

    # Tools exposed to OpenAI (always all tools; MCP is central)
    tools_for_call = TOOLS_FOR_OPENAI

    # Decide tool_choice
    if tool_choice_label == "auto":
        tool_choice = "auto"
    else:
        if tool_choice_label not in TOOL_NAMES:
            raise ValueError(f"Unknown tool_choice_label: {tool_choice_label}")
        tool_choice = {
            "type": "function",
            "function": {"name": tool_choice_label},
        }

    messages = [
        {
            "role": "system",
            "content": (
                "You are an enterprise assistant connected to a central "
                "Model Context Protocol (MCP) control plane. "
                "All tools are exposed via MCP; do not fabricate tool outputs. "
"Use `employee_lookup` with an `emp_id` integer (employees table columns: emp_id, name, dept, location, email). "
"Use `weather_lookup` for forecasts, even if the date is in the future."

            ),
        },
        {"role": "user", "content": user_msg},
    ]

    # First round: LLM may decide to call tools (or answer directly)
    resp = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools_for_call,
        tool_choice=tool_choice,
    )
    msg = resp.choices[0].message

    if getattr(msg, "content", None):
        log_lines.append(f"[LLM] {msg.content}")

    # If tools are used
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            tool_name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            log_lines.append(f"[TOOL CALL] {tool_name} -> {args}")
            result = mcp_call_tool(tool_name, args)
            log_lines.append(f"[TOOL RESULT] {tool_name} -> {result}")

            messages.append({
                "role": "assistant",
                "tool_calls": [{
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tool_name,
                        "arguments": json.dumps(args),
                    },
                }],
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": tool_name,
                "content": json.dumps(result),
            })

        # Second round: LLM sees tool outputs
        final = client.chat.completions.create(model=model_name, messages=messages)
        answer = final.choices[0].message.content or ""
        log_lines.append(f"[FINAL] {answer}")
    else:
        # No tools used
        answer = msg.content or ""
        log_lines.append(f"[FINAL no-tools] {answer}")

    debug_log = "\n".join(log_lines)
    if verbose:
        print(debug_log)
    return answer, debug_log


Quick Sanity Test (No UI Yet)

In [11]:
# Test: model 1, auto tools
ans, log = ask_with_model_and_tool(
    "Model 1 – gpt-4o" if not USE_AZURE else "Model 1 – HR (Azure dep1)",
    "auto",
    "Who is employee 101 and what is the Bengaluru weather on 2025-11-17?"
)
print("ANSWER:\n", ans)
print("\n--- DEBUG LOG ---\n", log)


[INFO] Using model: Model 1 – gpt-4o -> gpt-4o
[INFO] MCP tools available: ['employee_lookup', 'docs_search', 'weather_lookup']
[INFO] Tool choice: auto
[TOOL CALL] employee_lookup -> {'emp_id': 101}
[TOOL RESULT] employee_lookup -> {'emp_id': 101, 'name': 'Ravi', 'dept': 'AI', 'location': 'Bengaluru', 'email': 'ravi@example.com'}
[TOOL CALL] weather_lookup -> {'city': 'Bengaluru', 'date': '2025-11-17'}
[TOOL RESULT] weather_lookup -> {'city': 'Bengaluru', 'date': '2025-11-17', 'forecast': 'Light showers in evening'}
[FINAL] Employee 101 is Ravi, who works in the AI department and is located in Bengaluru. You can contact him at ravi@example.com. 

The weather forecast for Bengaluru on November 17, 2025, is expected to have light showers in the evening.
ANSWER:
 Employee 101 is Ravi, who works in the AI department and is located in Bengaluru. You can contact him at ravi@example.com. 

The weather forecast for Bengaluru on November 17, 2025, is expected to have light showers in the eveni

Gradio UI: Pick Model + Tool

This is the front-end.

In [12]:
import gradio as gr

MODEL_OPTIONS = list(MODEL_REGISTRY.keys())
TOOL_OPTIONS = ["auto"] + TOOL_NAMES

def gradio_chat(model_label, tool_choice_label, prompt):
    if not prompt.strip():
        return "Please enter a prompt.", ""
    answer, debug_log = ask_with_model_and_tool(
        model_label=model_label,
        tool_choice_label=tool_choice_label,
        user_msg=prompt,
        verbose=False,
    )
    return answer, debug_log

with gr.Blocks() as demo:
    gr.Markdown("## MCP Demo — 3 Models → 1 MCP → 3 Tools")
    gr.Markdown(
        "- **Model** dropdown = left side of your slide (different LLMs / deployments)\n"
        "- **Tool choice** = either `auto` (LLM decides) or a specific MCP tool\n"
        "- Under the hood: all models talk to the **same MCP registry** and call the **same tools**"
    )

    with gr.Row():
        model_dd = gr.Dropdown(
            MODEL_OPTIONS,
            value=MODEL_OPTIONS[0],
            label="Choose Model (LLM / Deployment)",
        )
        tool_dd = gr.Dropdown(
            TOOL_OPTIONS,
            value="auto",
            label="Tool choice (auto or specific MCP tool)",
        )

    prompt_tb = gr.Textbox(
        lines=3,
        label="Your prompt",
        placeholder="Ask something needing DB, Docs, or Weather...",
    )

    with gr.Row():
        answer_tb = gr.Textbox(lines=6, label="Model Answer")
        debug_tb = gr.Textbox(lines=12, label="Debug Log (tool calls via MCP)")

    send_btn = gr.Button("Send")

    send_btn.click(
        fn=gradio_chat,
        inputs=[model_dd, tool_dd, prompt_tb],
        outputs=[answer_tb, debug_tb],
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7bdbf84a866d749679.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
